# NingL 2022 CRC: SPECTRA analysis

Normal BMI range: $18.5 \leq BMI < 25$.

In [1]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings(
    'ignore', message=r'urllib3 .*does not match a supported version|urllib3 .*doesn.t match a supported version'
)

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import roc_auc_score

WORK_DIR = Path.cwd()
if not (WORK_DIR / '4. Name-converted relative abundance.csv').exists():
    raise FileNotFoundError('Run this notebook from its own directory.')

REPOSITORY_ROOT = next(
    candidate
    for path in (WORK_DIR, *WORK_DIR.parents)
    for candidate in (path, path / 'SPECTRA_GitHub_resource')
    if (candidate / 'models/metagenomic').exists()
    and (candidate / 'scripts/utils.py').exists()
)
utils_dir = REPOSITORY_ROOT / "scripts"
if str(utils_dir) not in sys.path:
    sys.path.insert(0, str(utils_dir))
from utils import predict_from_abundance_to_phenotype


## 1. Relative-abundance input

In [2]:
abundance = pd.read_csv(WORK_DIR / '4. Name-converted relative abundance.csv', index_col=0)
metadata = pd.read_csv(WORK_DIR / '0. metadata.csv', index_col=0)
metadata.index = metadata.index.astype(str)
abundance.index = abundance.index.astype(str)
metadata = metadata.loc[abundance.index]

assert abundance.shape == (116, 1685)
assert metadata['DatasetID'].eq('PRJNA731589').all()
assert metadata['BMI'].ge(18.5).all() and metadata['BMI'].lt(25).all()
assert metadata['true_label'].value_counts().to_dict() == {'HC': 64, 'CL': 52}

pd.DataFrame({
    'value': [abundance.shape[0], abundance.shape[1], 64, 52]
}, index=['samples', 'features', 'HC', 'CL'])

,value
samples,116
features,1685
HC,64
CL,52


## 2. SPECTRA prediction

In [3]:
model_dir = REPOSITORY_ROOT / 'models/metagenomic'
result = predict_from_abundance_to_phenotype(
    Abundance=abundance,
    MRI_model_path=model_dir / 'mri_calculators',
    SPECTRA_model_path=model_dir / 'spectra_model.pkl',
)

mri_score = result['MRI'].reindex(abundance.index)
probability = result['probability'].reindex(abundance.index)
mri_score.to_csv(WORK_DIR / '5. MRI scores.csv', float_format='%.17g')
probability.to_csv(WORK_DIR / '6. Probability.csv', float_format='%.17g')

display(mri_score.head())
display(probability.head())

,ACVD,AS,BloodPressureAbnormalities,ColorectalLesions,IBD,IGT,T2D,cirrhosis,control,fatty_liver,melanoma,schizophrenia
sample_id,,,,,,,,,,,,
SRR14610692,0.04,0.00,0.55,0.04,0.12,0.00,0.01,0.01,0.40,0.09,0.00,0.00
SRR14610575,0.10,0.05,0.03,0.04,0.57,0.00,0.13,0.03,0.13,0.00,0.00,0.02
SRR14610674,0.09,0.01,0.09,0.35,0.09,0.01,0.07,0.04,0.51,0.02,0.00,0.00
SRR14610584,0.08,0.00,0.23,0.42,0.13,0.00,0.03,0.00,0.52,0.02,0.01,0.00
SRR14610612,0.13,0.01,0.13,0.43,0.09,0.00,0.05,0.00,0.71,0.13,0.00,0.00


,ACVD,AS,BPA,CL,IBD,IGT,T2D,CI,HC,FL,ME,SC
sample_id,,,,,,,,,,,,
SRR14610692,0.003338,0.019766,0.735710,0.007267,0.029072,0.006463,0.000250,0.006746,0.179605,0.002579,0.000000,0.009204
SRR14610575,0.040778,0.021648,0.019213,0.080627,0.405242,0.014541,0.069352,0.009995,0.294213,0.036785,0.004461,0.003144
SRR14610674,0.015959,0.020817,0.043528,0.213271,0.036573,0.009629,0.037507,0.010650,0.595832,0.007257,0.005600,0.003377
SRR14610584,0.012294,0.018075,0.083874,0.362719,0.040980,0.002247,0.017275,0.002804,0.428508,0.013766,0.016304,0.001154
SRR14610612,0.030414,0.011936,0.018227,0.258151,0.023328,0.004222,0.018751,0.003209,0.612465,0.017462,0.000000,0.001836


## 3. Performance evaluation

In [4]:
case = metadata['true_label'].eq('CL').astype(int)
spectra_auc = roc_auc_score(case, probability['CL'])
pd.DataFrame({'AUC': [spectra_auc]}, index=['SPECTRA CL probability'])

,AUC
SPECTRA CL probability,0.711538


In [5]:
ranked_labels = np.argsort(-probability.to_numpy(), axis=1)[:, :3]
ranked_labels = probability.columns.to_numpy()[ranked_labels]

per_sample = metadata[['BMI', 'source_group', 'true_label']].copy()
for rank in range(3):
    per_sample[f'Rank{rank + 1} label'] = ranked_labels[:, rank]
for k in range(1, 4):
    per_sample[f'Top{k} correct'] = [
        true_label in labels[:k]
        for true_label, labels in zip(per_sample['true_label'], ranked_labels)
    ]

rows = []
for scope, mask in {
    'CL patients': per_sample['true_label'].eq('CL'),
    'HC': per_sample['true_label'].eq('HC'),
    'All samples': pd.Series(True, index=per_sample.index),
}.items():
    row = {'scope': scope, 'n': int(mask.sum())}
    for k in range(1, 4):
        row[f'Top{k} accuracy'] = per_sample.loc[mask, f'Top{k} correct'].mean()
    rows.append(row)

top_accuracy = pd.DataFrame(rows).set_index('scope')
top_accuracy

,n,Top1 accuracy,Top2 accuracy,Top3 accuracy
scope,,,,
CL patients,52,0.423077,0.711538,0.807692
HC,64,0.734375,0.984375,1.000000
All samples,116,0.594828,0.862069,0.913793


In [6]:
pd.DataFrame({
    'value': [
        spectra_auc,
        top_accuracy.loc['CL patients', 'Top1 accuracy'],
        top_accuracy.loc['CL patients', 'Top2 accuracy'],
        top_accuracy.loc['CL patients', 'Top3 accuracy'],
    ]
}, index=['SPECTRA AUC', 'CL Top1', 'CL Top2', 'CL Top3'])

,value
SPECTRA AUC,0.711538
CL Top1,0.423077
CL Top2,0.711538
CL Top3,0.807692
